# DINO-WM planning results

DINO-WM is compared with every method in the main planning evaluation using matched `H=1`, tasks, policy seeds, CEM settings, and evaluation budget. Results are mean ± STD over five seeds.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

EXPERIMENT_DIR = Path.cwd().resolve()
if not (EXPERIMENT_DIR / "aggregated_results.csv").exists():
    candidate = Path("planning/experiments/dino_wm").resolve()
    if candidate.exists():
        EXPERIMENT_DIR = candidate

PLANNING_ROOT = next(
    candidate
    for candidate in [EXPERIMENT_DIR, *EXPERIMENT_DIR.parents]
    if (candidate / "plot_style.py").exists() and (candidate / "experiments").is_dir()
)
sys.path.insert(0, str(PLANNING_ROOT))
from plot_style import FULL_WIDTH, apply_matplotlib_style, palette

apply_matplotlib_style()
ENV_ORDER = ["TwoRoom", "Reacher", "Push-T", "OGBench-Cube"]
METHOD_ORDER = ["Forward only", "SIGReg", "IDR", "DINO-WM", "Random"]
METHOD_COLORS = {
    "Forward only": palette["Med Grey"],
    "SIGReg": palette["Dark Red"],
    "IDR": palette["Dark Blue"],
    "DINO-WM": palette["Med Purple"],
    "Random": palette["Dark Grey"],
}


In [ ]:
dino = pd.read_csv(EXPERIMENT_DIR / "aggregated_results.csv")
dino = dino.query("status == 'ok'").copy()
assert len(dino) == 20, f"Expected 20 completed DINO-WM runs, found {len(dino)}"
assert dino.groupby("env_label").size().eq(5).all()
dino["display_method"] = "DINO-WM"
dino["repeat"] = dino["seed"]

main_path = EXPERIMENT_DIR.parent / "planning_eval" / "aggregated_results.csv"
main = pd.read_csv(main_path).query("status == 'ok'").copy()
assert len(main) == 80, f"Expected 80 completed main-evaluation rows, found {len(main)}"
main["display_method"] = main["method"].map({
    "forward_only": "Forward only",
    "sigreg": "SIGReg",
    "inverse": "IDR",
    "random": "Random",
})
main["repeat"] = main["seed_or_repeat"]

columns = ["env_label", "display_method", "repeat", "success_rate"]
comparison = pd.concat([main[columns], dino[columns]], ignore_index=True)
comparison.to_csv(EXPERIMENT_DIR / "comparison_results.csv", index=False)


In [ ]:
table = (
    comparison.groupby(["env_label", "display_method"], as_index=False)
    .agg(mean=("success_rate", "mean"), std=("success_rate", "std"), n=("success_rate", "size"))
)
table["env_label"] = pd.Categorical(table["env_label"], ENV_ORDER, ordered=True)
table["display_method"] = pd.Categorical(table["display_method"], METHOD_ORDER, ordered=True)
table = table.sort_values(["env_label", "display_method"]).reset_index(drop=True)
table.to_csv(EXPERIMENT_DIR / "comparison_summary_results.csv", index=False)
table

In [ ]:
fig, ax = plt.subplots(figsize=(FULL_WIDTH, 2.8))
x = np.arange(len(ENV_ORDER))
width = 0.16
offsets = np.arange(len(METHOD_ORDER)) - (len(METHOD_ORDER) - 1) / 2

for index, method in enumerate(METHOD_ORDER):
    rows = table[table["display_method"] == method].set_index("env_label").reindex(ENV_ORDER)
    ax.bar(
        x + offsets[index] * width,
        rows["mean"],
        width,
        yerr=rows["std"],
        capsize=1.5,
        label=method,
        color=METHOD_COLORS[method],
    )

ax.set_xticks(x, ENV_ORDER)
ax.set_ylabel("Planning success rate (%)")
ax.set_ylim(0, 105)
ax.grid(axis="y")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, 1.18))
fig.tight_layout()
fig.savefig(EXPERIMENT_DIR / "dino_wm_comparison.pdf")
plt.show()
